# 🛰️ Module 12.3 — RL for Satellite Image Analysis

*Reinforcement Learning for Image Processing — Real-World Project*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_12_Real_World_Projects/03_Satellite_Image_Analysis/satellite_image_analysis.ipynb)

---

## 🎯 Learning Objectives

1. Understand the unique challenges of satellite image processing (atmospheric interference, sensor noise, multi-spectral data).
2. Formulate satellite image enhancement as a Markov Decision Process with **classification-driven rewards**.
3. Generate realistic **synthetic satellite scenes** with land-use ground truth (vegetation, water, urban, bare soil).
4. Train a lightweight CNN classifier and use its confidence as a **reward signal** for the RL agent.
5. Implement a **DQN-based enhancement agent** that selects optimal processing actions for each satellite patch.
6. Evaluate per-class classification improvement and visualize land-use maps before and after RL enhancement.

In [ ]:
!pip install -q torch torchvision numpy matplotlib scikit-learn scikit-image

In [ ]:
# Download REAL open-source datasets for real-world projects
import torchvision
import subprocess
import sys

# MedMNIST for medical imaging (install + download)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist', '-q'])
import medmnist
from medmnist import ChestMNIST, DermaMNIST

# Chest X-rays (real medical images!)
chest_data = ChestMNIST(split='train', download=True, size=28)
print(f"✅ ChestMNIST: {len(chest_data)} real chest X-ray images")

# Dermatology images
derma_data = DermaMNIST(split='train', download=True, size=28)
print(f"✅ DermaMNIST: {len(derma_data)} real skin lesion images")

# EuroSAT for satellite imagery
try:
    eurosat = torchvision.datasets.EuroSAT(root='./data', download=True)
    print(f"✅ EuroSAT: {len(eurosat)} real satellite images (64x64, 10 land-use classes)")
except:
    print("⚠️ EuroSAT downloading...")

# CIFAR-10 for video/editing projects
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos for editing/video projects")

## Deep Derivation: RL for Satellite Image Analysis and Remote Sensing

### Step 1: Satellite Image Formation Model
The observed radiance at sensor:
$$L_{\text{sensor}}(\lambda) = \frac{\rho(\lambda) \cdot E_{\text{sun}}(\lambda) \cdot \cos\theta_s \cdot T_\downarrow(\lambda) \cdot T_\uparrow(\lambda)}{\pi} + L_{\text{path}}(\lambda)$$

where $\rho$ = surface reflectance, $E_{\text{sun}}$ = solar irradiance, $\theta_s$ = solar zenith angle, $T_\downarrow, T_\uparrow$ = atmospheric transmittance (down/up), $L_{\text{path}}$ = atmospheric scattering.

**Atmospheric correction (simplified):**
$$\rho(\lambda) = \frac{\pi (L_{\text{sensor}} - L_{\text{path}})}{E_{\text{sun}} \cos\theta_s \cdot T_\downarrow \cdot T_\uparrow}$$

**Derivation:** This follows from the radiative transfer equation. The key assumption is Lambertian reflectance (equal scattering in all directions). For non-Lambertian surfaces (water, snow), the BRDF must be modeled:
$$L = \frac{1}{\pi}\int_{\Omega_i} f_r(\omega_i, \omega_r) L_i(\omega_i) \cos\theta_i \, d\omega_i$$

where $f_r$ is the bidirectional reflectance distribution function. $\blacksquare$

### Step 2: Spectral Indices and Band Math
**Normalized Difference Vegetation Index (NDVI):**
$$\text{NDVI} = \frac{\rho_{\text{NIR}} - \rho_{\text{Red}}}{\rho_{\text{NIR}} + \rho_{\text{Red}}} \in [-1, 1]$$

**Derivation of why NDVI works:** Chlorophyll absorbs red light (660nm) and reflects NIR (860nm):
- Healthy vegetation: $\rho_{\text{NIR}} \gg \rho_{\text{Red}}$ → NDVI ≈ 0.6-0.9
- Bare soil: $\rho_{\text{NIR}} \approx \rho_{\text{Red}}$ → NDVI ≈ 0.1-0.2
- Water: $\rho_{\text{NIR}} < \rho_{\text{Red}}$ → NDVI < 0

**Modified indices for specific applications:**
$$\text{NDWI} = \frac{\rho_{\text{Green}} - \rho_{\text{NIR}}}{\rho_{\text{Green}} + \rho_{\text{NIR}}} \quad \text{(water detection)}$$
$$\text{NDBI} = \frac{\rho_{\text{SWIR}} - \rho_{\text{NIR}}}{\rho_{\text{SWIR}} + \rho_{\text{NIR}}} \quad \text{(built-up areas)}$$

**General form:** All normalized difference indices have the structure $\frac{a-b}{a+b}$, which is invariant to multiplicative illumination changes: $\frac{ka - kb}{ka + kb} = \frac{a-b}{a+b}$. $\blacksquare$

### Step 3: RL for Adaptive Band Selection
With $B$ spectral bands, selecting the optimal $k$ bands for a task is $\binom{B}{k}$ combinations. For Sentinel-2 ($B=13$, $k=4$): 715 combinations. For hyperspectral ($B=200$, $k=10$): $\sim 10^{16}$ combinations.

**RL formulation:**
- **State:** $s_t = (\text{selected bands}, \text{current classification accuracy})$
- **Action:** Add or remove a band
- **Reward:** $r_t = \Delta\text{accuracy} - \lambda \cdot \Delta\text{computation\_cost}$

**Derivation of mutual information criterion:**
$$I(Y; X_b | X_{b_1, \ldots, b_{t-1}}) = H(Y | X_{b_1, \ldots, b_{t-1}}) - H(Y | X_{b_1, \ldots, b_{t-1}}, X_b)$$

Select band $b$ that maximizes conditional MI. This is a greedy approximation to:
$$\max_{|S|=k} I(Y; X_S)$$

**Submodularity guarantee:** Mutual information is submodular, so greedy selection achieves at least $(1-1/e) \approx 63\%$ of optimal:
$$I(Y; X_{S_{\text{greedy}}}) \geq (1-1/e) \cdot I(Y; X_{S^*}) \quad \blacksquare$$

### Step 4: Change Detection with Temporal RL
Given multi-temporal images $\{I_{t_1}, I_{t_2}, \ldots, I_{t_n}\}$, detect changes.

**Multivariate Alteration Detection (MAD):**
Compute canonical correlations between $I_{t_1}$ and $I_{t_2}$:
$$\max_{a, b} \text{corr}(a^T I_{t_1}, b^T I_{t_2})$$

The MAD variates: $\text{MAD}_k = a_k^T I_{t_1} - b_k^T I_{t_2}$

**Chi-squared test for change:**
$$\chi^2 = \sum_{k=1}^p \frac{\text{MAD}_k^2}{\text{Var}(\text{MAD}_k)}$$

Under the null hypothesis (no change), $\chi^2 \sim \chi_p^2$.

**RL for sequential change monitoring:** The agent monitors a stream of images and decides when to trigger a change alarm:
$$\pi(a_t | s_t) = \sigma(W[f(I_{t}) - f(I_{t-1}); \chi_t^2; h_t] + b)$$

**CUSUM (Cumulative Sum) as optimal sequential test:**
$$S_t = \max(0, S_{t-1} + \log\frac{p_1(x_t)}{p_0(x_t)})$$

Alarm when $S_t > h$. By Wald's theorem, CUSUM minimizes worst-case detection delay:
$$\inf_\pi \sup_\nu E_\nu[T_\pi - \nu | T_\pi \geq \nu] \quad \text{s.t. } E_\infty[T_\pi] \geq \gamma \quad \blacksquare$$

### Step 5: Spatial Resolution Enhancement (Super-Resolution)
**Observation model:** $I_{\text{LR}} = D \cdot B \cdot I_{\text{HR}} + n$

where $D$ = downsampling, $B$ = blur kernel, $n$ = noise.

**MAP estimation:**
$$I_{\text{HR}}^* = \arg\max_{I} \left[\log p(I_{\text{LR}}|I) + \log p(I)\right]$$
$$= \arg\min_{I} \left[\|I_{\text{LR}} - DBI\|_2^2 + \lambda\Phi(I)\right]$$

**Derivation of the super-resolution RL agent:**
- **State:** $s_t = (I_t^{\text{SR}}, I_{\text{LR}}, \text{upscale\_factor})$
- **Action:** Select super-resolution strategy for each patch
- **Reward:** Weighted combination:

$$R = w_1 \cdot \text{PSNR}(I^{\text{SR}}, I^{\text{HR}}) + w_2 \cdot \text{SSIM}(I^{\text{SR}}, I^{\text{HR}}) - w_3 \cdot \text{spectral\_distortion}$$

**Spectral angle mapper (SAM):**
$$\text{SAM}(x, y) = \arccos\frac{x \cdot y}{\|x\| \|y\|}$$

SAM $= 0$ means perfect spectral fidelity. This is critical for satellite imagery where spectral accuracy determines downstream analysis quality. $\blacksquare$

### Step 6: Object Detection in Satellite Imagery (Rotated Boxes)
Objects in aerial images can be at any orientation. Use oriented bounding boxes:
$$B = (x_c, y_c, w, h, \theta)$$

**Rotated IoU computation:** Transform to polygon intersection:
$$\text{IoU}_{\text{rot}} = \frac{|P_1 \cap P_2|}{|P_1 \cup P_2|}$$

where $P_1, P_2$ are rotated rectangles. The intersection is computed via the Sutherland-Hodgman polygon clipping algorithm in $O(n \cdot m)$ where $n, m$ are vertex counts.

**Derivation of rotation-equivariant features:** A feature $\phi$ is rotation-equivariant if:
$$\phi(R_\theta \cdot I) = \rho(\theta) \cdot \phi(I)$$

where $R_\theta$ rotates the image and $\rho$ is a representation of $SO(2)$. Steerable CNNs achieve this by constraining filters to be linear combinations of circular harmonics:
$$f(r, \psi) = \sum_{m=-M}^{M} w_m(r) e^{im\psi}$$

This ensures the network's predictions rotate with the input, critical for orientation-dependent detection. $\blacksquare$

### Step 7: Multi-Resolution RL Policy for Large-Scale Analysis
Satellite images are enormous ($10000 \times 10000+$ pixels). The RL agent operates hierarchically:

**Coarse level:** $I_{\text{coarse}} = \text{downsample}(I, 16\times)$, decide which regions are interesting
**Fine level:** $I_{\text{fine}} = \text{crop}(I, \text{region})$, perform detailed analysis

**Computational savings:**
$$\text{Cost}_{\text{hierarchical}} = \frac{H \cdot W}{16^2} \cdot C_{\text{coarse}} + K \cdot \frac{H \cdot W}{16^2} \cdot C_{\text{fine}}$$

vs. $\text{Cost}_{\text{full}} = H \cdot W \cdot C_{\text{fine}}$

**Speedup ratio:**
$$\frac{\text{Cost}_{\text{full}}}{\text{Cost}_{\text{hierarchical}}} = \frac{256}{C_{\text{coarse}}/C_{\text{fine}} + K}$$

For $K = 10$ interesting regions and $C_{\text{coarse}}/C_{\text{fine}} = 0.1$: speedup $\approx 25\times$.

**Derivation of optimal attention budget $K$:** Given a total compute budget $B$:
$$K^* = \arg\max_K \text{Recall}(K) \quad \text{s.t.} \quad C_{\text{coarse}} + K \cdot C_{\text{fine}} \leq B$$

By the diminishing returns of attention (more regions = less compute per region), the optimal $K$ balances breadth and depth of analysis. $\blacksquare$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
from sklearn.metrics import confusion_matrix, classification_report
import random
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All imports successful ✓')

---
## 1. Introduction — Satellite Image Processing Challenges

Satellite imagery is the backbone of remote sensing applications: **land-use mapping**, **urban planning**, **crop monitoring**, **disaster response**, and **climate science**. However, raw satellite images suffer from several degradation sources:

| Degradation | Cause | Effect |
|---|---|---|
| **Atmospheric haze** | Aerosol scattering | Reduced contrast, colour shift |
| **Cloud shadows** | Partial cloud cover | Dark patches, lost detail |
| **Sensor noise** | Thermal / electronic noise | Speckle, salt-and-pepper artifacts |
| **Low contrast** | Poor illumination geometry | Flat histogram, poor class separation |

Traditional image enhancement applies a **fixed pipeline** (atmospheric correction → dehazing → sharpening). This fails because:
- Different regions in the **same** scene need different processing (e.g., water vs. urban).
- Degradation severity varies with weather, sun angle, and sensor age.

### Why Reinforcement Learning?

An RL agent can **adaptively select** the best enhancement action for each image patch by:
1. Observing the current state of the patch.
2. Choosing from a rich action set (atmospheric correction, vegetation enhancement, etc.).
3. Receiving reward proportional to **downstream classification accuracy** — directly optimising the end goal.

---
## 2. Problem Formulation — MDP for Satellite Image Enhancement

We cast adaptive satellite image enhancement as a Markov Decision Process $\mathcal{M} = (\mathcal{S}, \mathcal{A}, R, T, \gamma)$.

### 2.1 State Space $\mathcal{S}$

Each state is a satellite image patch $\mathbf{I} \in \mathbb{R}^{H \times W \times C}$ where $C$ may include multi-spectral bands. We extract compact features:

$$
s = \left[\mu_c, \sigma_c, \text{skew}_c, \text{kurt}_c, E_{\text{edge}}, \text{NDVI}_{\text{proxy}}\right]_{c=1}^{C}
$$

where $\mu_c, \sigma_c$ are per-channel mean and standard deviation, and $E_{\text{edge}}$ captures edge energy via Sobel magnitude.

### 2.2 Action Space $\mathcal{A}$

The agent selects from 10 image processing actions:

$$
\mathcal{A} = \{\texttt{atmospheric\_correction},\; \texttt{dehazing},\; \texttt{contrast\_stretch},\; \texttt{histogram\_matching},\; \texttt{vegetation\_enhance},\; \texttt{water\_enhance},\; \texttt{urban\_enhance},\; \texttt{noise\_filter},\; \texttt{sharpen},\; \texttt{no\_op}\}
$$

### 2.3 NDVI — Normalized Difference Vegetation Index

For multi-spectral imagery with Near-Infrared (NIR) and Red bands:

$$
\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}
$$

NDVI $\in [-1, 1]$: values near $+1$ indicate dense vegetation; values near $0$ indicate bare soil or urban areas; negative values indicate water.

For RGB-only images we use the **Green-Red proxy**:

$$
\text{NDVI}_{\text{proxy}} = \frac{G - R}{G + R + \epsilon}
$$

### 2.4 Reward Function $R$

The reward incentivises enhancement that **improves classification**:

$$
R = \alpha \cdot \text{ClassificationAccuracy}(I_{\text{enhanced}}) + \beta \cdot \text{SSIM}(I_{\text{enhanced}}, I_{\text{clean}}) - \gamma \cdot \text{DistortionPenalty}
$$

where:
- $\text{ClassificationAccuracy}$: confidence of a pre-trained land-use classifier on the enhanced image.
- $\text{SSIM}$: Structural Similarity Index ensuring the enhanced image remains faithful.
- $\text{DistortionPenalty}$: penalises excessive pixel value changes.

We set $\alpha = 0.6$, $\beta = 0.3$, $\gamma = 0.1$ to prioritise classification performance.

### 2.5 Transition & Episode Structure

Each episode processes one satellite patch through up to $T_{\max} = 5$ sequential enhancement steps. The transition is deterministic: applying action $a$ transforms the current image state:

$$
s_{t+1} = f_a(s_t)
$$

The episode terminates when the agent selects $\texttt{no\_op}$ or the step budget is exhausted.

---
## 3. Synthetic Satellite Image Generation

We generate realistic synthetic satellite scenes with **four land-use classes** and controllable degradation.

In [ ]:
# --- Land-use class definitions ---
CLASSES = ['vegetation', 'water', 'urban', 'bare_soil']
NUM_CLASSES = len(CLASSES)

CLASS_COLORS_RGB = {
    'vegetation': np.array([34, 139, 34]) / 255.0,
    'water':      np.array([30, 80, 180]) / 255.0,
    'urban':      np.array([160, 160, 165]) / 255.0,
    'bare_soil':  np.array([180, 140, 100]) / 255.0,
}

CLASS_CMAP = mcolors.ListedColormap(
    [CLASS_COLORS_RGB[c] for c in CLASSES]
)

PATCH_SIZE = 64


def generate_land_use_map(size=64, num_regions=8, rng=None):
    """Create a random land-use ground truth map with Voronoi-like regions."""
    if rng is None:
        rng = np.random.default_rng()
    label_map = np.zeros((size, size), dtype=np.int64)
    centres = rng.integers(0, size, size=(num_regions, 2))
    classes = rng.integers(0, NUM_CLASSES, size=num_regions)
    yy, xx = np.mgrid[:size, :size]
    for idx in range(num_regions):
        cy, cx = centres[idx]
        dist = (yy - cy) ** 2 + (xx - cx) ** 2
        if idx == 0:
            min_dist = dist.copy()
            label_map[:] = classes[idx]
        else:
            mask = dist < min_dist
            label_map[mask] = classes[idx]
            min_dist[mask] = dist[mask]
    return label_map


def label_map_to_image(label_map, rng=None):
    """Convert a label map to a clean RGB satellite image with texture."""
    if rng is None:
        rng = np.random.default_rng()
    h, w = label_map.shape
    img = np.zeros((h, w, 3), dtype=np.float32)
    for idx, cls_name in enumerate(CLASSES):
        mask = label_map == idx
        base_color = CLASS_COLORS_RGB[cls_name].astype(np.float32)
        count = mask.sum()
        if count == 0:
            continue
        texture = rng.normal(0, 0.04, size=(count, 3)).astype(np.float32)
        img[mask] = base_color + texture
    return np.clip(img, 0.0, 1.0)


def apply_atmospheric_degradation(img, severity=0.5, rng=None):
    """Apply haze, cloud shadows, and sensor noise."""
    if rng is None:
        rng = np.random.default_rng()
    degraded = img.copy()

    # Atmospheric haze: additive white-blue tint
    haze_strength = severity * rng.uniform(0.15, 0.35)
    haze_color = np.array([0.7, 0.75, 0.85], dtype=np.float32)
    h, w = img.shape[:2]
    haze_map = rng.uniform(0.5, 1.0, size=(h // 4, w // 4)).astype(np.float32)
    from scipy.ndimage import zoom
    haze_map = zoom(haze_map, (h / (h // 4), w / (w // 4)), order=1)
    haze_map = haze_map[:h, :w]
    for c in range(3):
        degraded[:, :, c] += haze_strength * haze_color[c] * haze_map

    # Cloud shadow: dark elliptical region
    if rng.random() > 0.3:
        cy, cx = rng.integers(h // 4, 3 * h // 4), rng.integers(w // 4, 3 * w // 4)
        ry, rx = rng.integers(8, h // 3), rng.integers(8, w // 3)
        yy, xx = np.mgrid[:h, :w]
        shadow_mask = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1.0
        degraded[shadow_mask] *= (1.0 - severity * 0.4)

    # Sensor noise
    noise_std = severity * 0.06
    degraded += rng.normal(0, noise_std, degraded.shape).astype(np.float32)

    # Contrast reduction
    degraded = degraded * (1.0 - severity * 0.2) + severity * 0.1

    return np.clip(degraded, 0.0, 1.0)


def generate_dataset(n_samples, size=64, severity_range=(0.3, 0.8), seed=42):
    """Generate a complete synthetic satellite dataset."""
    rng = np.random.default_rng(seed)
    clean_images = []
    degraded_images = []
    label_maps = []

    for _ in range(n_samples):
        lmap = generate_land_use_map(size, num_regions=rng.integers(5, 12), rng=rng)
        clean = label_map_to_image(lmap, rng=rng)
        sev = rng.uniform(*severity_range)
        degraded = apply_atmospheric_degradation(clean, severity=sev, rng=rng)
        clean_images.append(clean)
        degraded_images.append(degraded)
        label_maps.append(lmap)

    return (
        np.array(clean_images),
        np.array(degraded_images),
        np.array(label_maps),
    )


print('Satellite data generators defined ✓')

In [ ]:
# --- Generate datasets ---
train_clean, train_degraded, train_labels = generate_dataset(400, seed=42)
val_clean, val_degraded, val_labels = generate_dataset(100, seed=123)

print(f'Training set:   {train_clean.shape[0]} images, shape {train_clean.shape[1:]}')
print(f'Validation set: {val_clean.shape[0]} images, shape {val_clean.shape[1:]}')

In [ ]:
# --- Visualise synthetic satellite scenes ---
fig, axes = plt.subplots(3, 6, figsize=(20, 10))

for col in range(6):
    idx = col * 10
    axes[0, col].imshow(train_clean[idx])
    axes[0, col].set_title(f'Clean #{idx}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(train_degraded[idx])
    axes[1, col].set_title('Degraded', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(train_labels[idx], cmap=CLASS_CMAP,
                        vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
    axes[2, col].set_title('Ground Truth', fontsize=10)
    axes[2, col].axis('off')

legend_elements = [Patch(facecolor=CLASS_COLORS_RGB[c], label=c.replace('_', ' ').title())
                   for c in CLASSES]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11,
           frameon=True, fancybox=True)

fig.suptitle('Synthetic Satellite Scenes — Clean vs Degraded vs Ground Truth',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.1 Degradation Analysis

Let us quantify how atmospheric degradation affects spectral separability — the key factor for land-use classification.

In [ ]:
# --- Per-class spectral statistics: Clean vs Degraded ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
channel_names = ['Red', 'Green', 'Blue']

for ch_idx, (ax, ch_name) in enumerate(zip(axes, channel_names)):
    for cls_idx, cls_name in enumerate(CLASSES):
        clean_vals = train_clean[train_labels == cls_idx, ch_idx].ravel()
        deg_vals = train_degraded[train_labels == cls_idx, ch_idx].ravel()
        clean_sub = np.random.choice(clean_vals, size=min(5000, len(clean_vals)), replace=False)
        deg_sub = np.random.choice(deg_vals, size=min(5000, len(deg_vals)), replace=False)
        color = CLASS_COLORS_RGB[cls_name]
        ax.hist(clean_sub, bins=40, alpha=0.5, density=True, color=color,
                label=f'{cls_name} (clean)' if ch_idx == 0 else '')
        ax.hist(deg_sub, bins=40, alpha=0.3, density=True, color=color,
                linestyle='--', histtype='step', linewidth=2,
                label=f'{cls_name} (degraded)' if ch_idx == 0 else '')
    ax.set_title(f'{ch_name} Channel Distribution', fontsize=12)
    ax.set_xlabel('Pixel Intensity')
    ax.set_ylabel('Density')

axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle('Spectral Overlap Increases After Atmospheric Degradation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Land-Use Classifier — Reward Signal Source

We train a lightweight **CNN classifier** on clean satellite patches. Its prediction confidence on enhanced images drives the RL reward. The classifier maps a $64\times64\times3$ patch to one of 4 classes.

### Pixel-Level Classifier Architecture

We use a small **fully-convolutional network** that outputs per-pixel class probabilities:

$$
p(y_{ij} = k \mid \mathbf{I}) = \text{softmax}\left(f_{\theta}(\mathbf{I})\right)_{ijk}
$$

The classification accuracy for the entire patch is the fraction of correctly classified pixels.

In [ ]:
class PixelClassifier(nn.Module):
    """Small fully-convolutional network for per-pixel land-use classification."""

    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.head = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        return self.head(self.features(x))


def images_to_tensor(images):
    """(N, H, W, 3) float32 numpy → (N, 3, H, W) float32 tensor."""
    return torch.from_numpy(images.transpose(0, 3, 1, 2)).float()


def compute_pixel_accuracy(model, images, labels, batch_size=64):
    """Compute per-pixel classification accuracy."""
    model.eval()
    correct = 0
    total = 0
    img_t = images_to_tensor(images)
    lab_t = torch.from_numpy(labels).long()
    with torch.no_grad():
        for i in range(0, len(img_t), batch_size):
            x = img_t[i:i+batch_size].to(device)
            y = lab_t[i:i+batch_size].to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total


print('Classifier architecture defined ✓')

In [ ]:
# --- Train the pixel classifier on CLEAN images ---
classifier = PixelClassifier(NUM_CLASSES).to(device)
cls_optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
cls_criterion = nn.CrossEntropyLoss()

train_img_t = images_to_tensor(train_clean).to(device)
train_lab_t = torch.from_numpy(train_labels).long().to(device)

cls_losses = []
cls_accuracies = []
NUM_CLS_EPOCHS = 20
BATCH_SIZE = 32

for epoch in range(NUM_CLS_EPOCHS):
    classifier.train()
    epoch_loss = 0.0
    n_batches = 0
    indices = np.random.permutation(len(train_img_t))

    for i in range(0, len(indices), BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        x = train_img_t[batch_idx]
        y = train_lab_t[batch_idx]
        logits = classifier(x)
        loss = cls_criterion(logits, y)
        cls_optimizer.zero_grad()
        loss.backward()
        cls_optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    cls_losses.append(avg_loss)

    val_acc = compute_pixel_accuracy(classifier, val_clean, val_labels)
    cls_accuracies.append(val_acc)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_CLS_EPOCHS}  '
              f'Loss: {avg_loss:.4f}  Val Accuracy: {val_acc:.4f}')

# Move training data back to CPU to free GPU memory
train_img_t = train_img_t.cpu()
train_lab_t = train_lab_t.cpu()

print(f'\nFinal classifier accuracy on clean images: {cls_accuracies[-1]:.4f}')

In [ ]:
# --- Classifier training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_CLS_EPOCHS + 1), cls_losses, 'b-o', markersize=4, linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('Classifier Training Loss', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, NUM_CLS_EPOCHS + 1), cls_accuracies, 'g-s', markersize=4, linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Pixel Accuracy', fontsize=12)
ax2.set_title('Classifier Validation Accuracy (Clean Images)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show accuracy on degraded images (baseline)
degraded_acc = compute_pixel_accuracy(classifier, val_degraded, val_labels)
print(f'Classifier accuracy on CLEAN val images:    {cls_accuracies[-1]:.4f}')
print(f'Classifier accuracy on DEGRADED val images: {degraded_acc:.4f}')
print(f'Accuracy drop due to degradation:           {cls_accuracies[-1] - degraded_acc:.4f}')

---
## 5. Implementation — RL Enhancement Pipeline

### 5.1 Enhancement Actions

Each action is a differentiable (or at least deterministic) image transformation.

In [ ]:
from scipy.ndimage import gaussian_filter, uniform_filter


def action_atmospheric_correction(img):
    """Subtract estimated haze using dark channel prior approximation."""
    dark = np.min(img, axis=2)
    atm_light = np.percentile(dark, 99)
    transmission = 1.0 - 0.6 * (dark / (atm_light + 1e-6))
    transmission = np.clip(transmission, 0.3, 1.0)
    result = np.zeros_like(img)
    for c in range(3):
        result[:, :, c] = (img[:, :, c] - atm_light * (1 - transmission)) / (transmission + 1e-6)
    return np.clip(result, 0, 1)


def action_dehazing(img):
    """Simple dehazing via Retinex-inspired log transform."""
    log_img = np.log1p(img * 255) / np.log(256)
    blurred = gaussian_filter(log_img, sigma=8)
    enhanced = log_img - 0.4 * blurred + 0.4
    return np.clip(enhanced, 0, 1).astype(np.float32)


def action_contrast_stretch(img):
    """Per-channel percentile-based contrast stretch."""
    result = np.zeros_like(img)
    for c in range(3):
        p2, p98 = np.percentile(img[:, :, c], [2, 98])
        result[:, :, c] = (img[:, :, c] - p2) / (p98 - p2 + 1e-6)
    return np.clip(result, 0, 1).astype(np.float32)


def action_histogram_matching(img):
    """Match histogram to a uniform target distribution."""
    result = np.zeros_like(img)
    for c in range(3):
        values = img[:, :, c].ravel()
        order = np.argsort(values)
        target = np.linspace(0.05, 0.95, len(values))
        result_flat = np.zeros_like(values)
        result_flat[order] = target
        result[:, :, c] = result_flat.reshape(img.shape[:2])
    return result.astype(np.float32)


def action_vegetation_enhance(img):
    """Boost green channel relative to red to enhance vegetation contrast."""
    result = img.copy()
    ndvi_proxy = (img[:, :, 1] - img[:, :, 0]) / (img[:, :, 1] + img[:, :, 0] + 1e-6)
    veg_mask = ndvi_proxy > 0.05
    result[veg_mask, 1] = np.clip(result[veg_mask, 1] * 1.25, 0, 1)
    result[veg_mask, 0] = np.clip(result[veg_mask, 0] * 0.85, 0, 1)
    return result.astype(np.float32)


def action_water_enhance(img):
    """Boost blue channel and suppress red/green for water bodies."""
    result = img.copy()
    water_idx = (img[:, :, 2] > img[:, :, 0]) & (img[:, :, 2] > img[:, :, 1] * 0.9)
    result[water_idx, 2] = np.clip(result[water_idx, 2] * 1.2, 0, 1)
    result[water_idx, 0] = np.clip(result[water_idx, 0] * 0.85, 0, 1)
    return result.astype(np.float32)


def action_urban_enhance(img):
    """Enhance edges and gray-level contrast for urban features."""
    gray = np.mean(img, axis=2)
    sharpened_gray = gray + 0.4 * (gray - uniform_filter(gray, size=3))
    urban_mask = np.abs(img[:, :, 0] - img[:, :, 1]) < 0.08
    result = img.copy()
    for c in range(3):
        result[urban_mask, c] = np.clip(
            result[urban_mask, c] + 0.15 * (sharpened_gray[urban_mask] - gray[urban_mask]),
            0, 1
        )
    return result.astype(np.float32)


def action_noise_filter(img):
    """Bilateral-style noise reduction via guided Gaussian filter."""
    return gaussian_filter(img, sigma=(0.8, 0.8, 0)).astype(np.float32)


def action_sharpen(img):
    """Unsharp masking."""
    blurred = gaussian_filter(img, sigma=(1.2, 1.2, 0))
    result = img + 0.5 * (img - blurred)
    return np.clip(result, 0, 1).astype(np.float32)


def action_no_op(img):
    return img.copy()


ACTIONS = [
    action_atmospheric_correction,
    action_dehazing,
    action_contrast_stretch,
    action_histogram_matching,
    action_vegetation_enhance,
    action_water_enhance,
    action_urban_enhance,
    action_noise_filter,
    action_sharpen,
    action_no_op,
]

ACTION_NAMES = [
    'atmos_correct', 'dehaze', 'contrast_stretch', 'hist_match',
    'veg_enhance', 'water_enhance', 'urban_enhance',
    'noise_filter', 'sharpen', 'no_op',
]

NUM_ACTIONS = len(ACTIONS)
print(f'Defined {NUM_ACTIONS} enhancement actions ✓')

In [ ]:
# --- Visualise each action applied to a sample degraded image ---
sample_idx = 5
sample_deg = train_degraded[sample_idx]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (action_fn, action_name) in enumerate(zip(ACTIONS, ACTION_NAMES)):
    ax = axes[i // 5, i % 5]
    result = action_fn(sample_deg)
    ax.imshow(result)
    ax.set_title(action_name, fontsize=10, fontweight='bold')
    ax.axis('off')

fig.suptitle('Effect of Each Enhancement Action on a Degraded Satellite Patch',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Satellite Image Environment

The environment wraps the enhancement loop: the agent observes image features, selects an action, and receives reward from the classifier.

In [ ]:
def compute_ssim_simple(img1, img2):
    """Simplified SSIM between two images."""
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu1, mu2 = img1.mean(), img2.mean()
    s1, s2 = img1.var(), img2.var()
    s12 = ((img1 - mu1) * (img2 - mu2)).mean()
    ssim = ((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / \
           ((mu1 ** 2 + mu2 ** 2 + C1) * (s1 + s2 + C2))
    return float(ssim)


def extract_state_features(img):
    """Extract compact feature vector from an image patch."""
    features = []
    for c in range(3):
        ch = img[:, :, c]
        features.extend([
            ch.mean(), ch.std(),
            float(np.median(ch)),
            float(np.percentile(ch, 10)),
            float(np.percentile(ch, 90)),
        ])
    # NDVI proxy
    ndvi = (img[:, :, 1] - img[:, :, 0]) / (img[:, :, 1] + img[:, :, 0] + 1e-6)
    features.extend([ndvi.mean(), ndvi.std()])
    # Edge energy (Sobel approximation)
    gray = np.mean(img, axis=2)
    dx = np.diff(gray, axis=1)
    dy = np.diff(gray, axis=0)
    edge_energy = (dx ** 2).mean() + (dy ** 2).mean()
    features.append(edge_energy)
    # Entropy approximation
    hist, _ = np.histogram(gray, bins=32, range=(0, 1))
    hist = hist / hist.sum()
    hist = hist[hist > 0]
    entropy = -np.sum(hist * np.log2(hist))
    features.append(entropy)
    return np.array(features, dtype=np.float32)


STATE_DIM = len(extract_state_features(train_degraded[0]))
print(f'State feature dimension: {STATE_DIM}')


class SatelliteEnhancementEnv:
    """RL environment for sequential satellite image enhancement."""

    def __init__(self, degraded_images, clean_images, label_maps,
                 classifier_model, max_steps=5,
                 alpha=0.6, beta=0.3, gamma_penalty=0.1):
        self.degraded_images = degraded_images
        self.clean_images = clean_images
        self.label_maps = label_maps
        self.classifier = classifier_model
        self.max_steps = max_steps
        self.alpha = alpha
        self.beta = beta
        self.gamma_penalty = gamma_penalty
        self.current_idx = 0
        self.current_image = None
        self.original_degraded = None
        self.clean_reference = None
        self.current_labels = None
        self.step_count = 0
        self.action_history = []

    def reset(self, idx=None):
        if idx is None:
            idx = np.random.randint(len(self.degraded_images))
        self.current_idx = idx
        self.current_image = self.degraded_images[idx].copy()
        self.original_degraded = self.degraded_images[idx].copy()
        self.clean_reference = self.clean_images[idx].copy()
        self.current_labels = self.label_maps[idx]
        self.step_count = 0
        self.action_history = []
        return extract_state_features(self.current_image)

    def _classify_accuracy(self, img):
        """Run classifier and return pixel accuracy."""
        self.classifier.eval()
        with torch.no_grad():
            x = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0).to(device)
            logits = self.classifier(x)
            preds = logits.argmax(dim=1).squeeze().cpu().numpy()
        acc = (preds == self.current_labels).mean()
        return float(acc)

    def step(self, action):
        prev_acc = self._classify_accuracy(self.current_image)
        self.current_image = ACTIONS[action](self.current_image)
        self.action_history.append(action)
        self.step_count += 1

        new_acc = self._classify_accuracy(self.current_image)
        ssim = compute_ssim_simple(self.current_image, self.clean_reference)
        distortion = np.mean((self.current_image - self.original_degraded) ** 2)

        reward = (self.alpha * (new_acc - prev_acc) * 10 +
                  self.beta * ssim -
                  self.gamma_penalty * distortion)

        done = (action == NUM_ACTIONS - 1) or (self.step_count >= self.max_steps)
        next_state = extract_state_features(self.current_image)

        info = {
            'accuracy': new_acc,
            'ssim': ssim,
            'distortion': distortion,
            'prev_accuracy': prev_acc,
        }
        return next_state, reward, done, info


print('SatelliteEnhancementEnv defined ✓')

### 5.3 DQN Agent

We implement a **Deep Q-Network** with experience replay and a target network:

$$
\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[\left(r + \gamma \max_{a'} Q_{\theta^{-}}(s', a') - Q_\theta(s, a)\right)^2\right]
$$

where $\theta^{-}$ are target network parameters updated periodically.

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


class DQN(nn.Module):
    def __init__(self, state_dim, num_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, num_actions),
        )

    def forward(self, x):
        return self.net(x)


class DQNAgent:
    def __init__(self, state_dim, num_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=500,
                 buffer_size=10000, batch_size=64, target_update=50):
        self.num_actions = num_actions
        self.gamma = gamma
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps_done = 0

        self.policy_net = DQN(state_dim, num_actions).to(device)
        self.target_net = DQN(state_dim, num_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)

    def get_epsilon(self):
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
               np.exp(-self.steps_done / self.epsilon_decay)

    def select_action(self, state, training=True):
        if training:
            eps = self.get_epsilon()
            self.steps_done += 1
            if random.random() < eps:
                return random.randrange(self.num_actions)

        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_t)
            return q_values.argmax(dim=1).item()

    def update(self):
        if len(self.buffer) < self.batch_size:
            return 0.0

        batch = self.buffer.sample(self.batch_size)
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).unsqueeze(1).to(device)

        q_values = self.policy_net(states).gather(1, actions)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1, keepdim=True)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)

        loss = F.smooth_l1_loss(q_values, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def sync_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())


print('DQN Agent defined ✓')

### 5.4 Training Loop

We train the DQN agent to maximise the classification-oriented reward.

In [ ]:
# --- Training ---
env = SatelliteEnhancementEnv(
    train_degraded, train_clean, train_labels,
    classifier, max_steps=5
)

agent = DQNAgent(
    state_dim=STATE_DIM,
    num_actions=NUM_ACTIONS,
    lr=5e-4,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=400,
    buffer_size=8000,
    batch_size=64,
    target_update=30,
)

NUM_EPISODES = 600
episode_rewards = []
episode_accuracies = []
episode_losses = []
action_counts = np.zeros(NUM_ACTIONS)

for ep in range(NUM_EPISODES):
    state = env.reset()
    total_reward = 0.0
    ep_loss = 0.0
    done = False
    steps = 0

    while not done:
        action = agent.select_action(state, training=True)
        next_state, reward, done, info = env.step(action)
        agent.buffer.push(state, action, reward, next_state, float(done))
        loss = agent.update()
        ep_loss += loss
        total_reward += reward
        action_counts[action] += 1
        state = next_state
        steps += 1

    episode_rewards.append(total_reward)
    episode_accuracies.append(info['accuracy'])
    episode_losses.append(ep_loss / max(steps, 1))

    if (ep + 1) % agent.target_update == 0:
        agent.sync_target()

    if (ep + 1) % 100 == 0:
        avg_r = np.mean(episode_rewards[-100:])
        avg_a = np.mean(episode_accuracies[-100:])
        eps = agent.get_epsilon()
        print(f'Episode {ep+1:4d}/{NUM_EPISODES}  '
              f'Avg Reward: {avg_r:+.3f}  '
              f'Avg Accuracy: {avg_a:.4f}  '
              f'Epsilon: {eps:.3f}')

print('\nTraining complete ✓')

---
## 6. Results & Analysis

In [ ]:
# --- 6.1 Training Curves ---
def smooth(data, window=20):
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Reward curve
axes[0].plot(smooth(episode_rewards), color='#2196F3', linewidth=2)
axes[0].set_title('Episode Reward (smoothed)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(smooth(episode_accuracies), color='#4CAF50', linewidth=2)
axes[1].axhline(y=degraded_acc, color='r', linestyle='--', alpha=0.7,
                label=f'Degraded baseline ({degraded_acc:.3f})')
axes[1].set_title('Classification Accuracy After Enhancement', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Pixel Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss curve
axes[2].plot(smooth(episode_losses), color='#FF5722', linewidth=2)
axes[2].set_title('DQN Loss (smoothed)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Loss')
axes[2].grid(True, alpha=0.3)

fig.suptitle('RL Agent Training Progress', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.2 Action Distribution ---
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_ACTIONS))
bars = ax.bar(range(NUM_ACTIONS), action_counts, color=colors, edgecolor='gray')
ax.set_xticks(range(NUM_ACTIONS))
ax.set_xticklabels(ACTION_NAMES, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('Times Selected', fontsize=12)
ax.set_title('Action Selection Distribution During Training', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, count in zip(bars, action_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{int(count)}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- 6.3 Before/After Enhancement on Validation Set ---
val_env = SatelliteEnhancementEnv(
    val_degraded, val_clean, val_labels,
    classifier, max_steps=5
)

val_enhanced = []
val_action_sequences = []
val_acc_before = []
val_acc_after = []

for i in range(len(val_degraded)):
    state = val_env.reset(idx=i)
    acc_before = val_env._classify_accuracy(val_env.current_image)
    val_acc_before.append(acc_before)
    done = False
    actions_taken = []

    while not done:
        action = agent.select_action(state, training=False)
        state, _, done, info = val_env.step(action)
        actions_taken.append(action)

    val_enhanced.append(val_env.current_image.copy())
    val_action_sequences.append(actions_taken)
    val_acc_after.append(info['accuracy'])

val_enhanced = np.array(val_enhanced)
val_acc_before = np.array(val_acc_before)
val_acc_after = np.array(val_acc_after)

print(f'Average accuracy BEFORE enhancement: {val_acc_before.mean():.4f}')
print(f'Average accuracy AFTER enhancement:  {val_acc_after.mean():.4f}')
print(f'Average improvement:                 {(val_acc_after - val_acc_before).mean():.4f}')

In [ ]:
# --- 6.4 Visual Before/After Comparison ---
fig, axes = plt.subplots(4, 6, figsize=(22, 15))
display_indices = [0, 10, 25, 40, 55, 70]

row_labels = ['Degraded', 'RL Enhanced', 'Clean (Reference)', 'Ground Truth']

for col, idx in enumerate(display_indices):
    axes[0, col].imshow(val_degraded[idx])
    axes[0, col].set_title(f'Acc: {val_acc_before[idx]:.3f}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(val_enhanced[idx])
    seq = ' → '.join([ACTION_NAMES[a][:6] for a in val_action_sequences[idx]])
    axes[1, col].set_title(f'Acc: {val_acc_after[idx]:.3f}', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(val_clean[idx])
    clean_acc = compute_pixel_accuracy(
        classifier, val_clean[idx:idx+1], val_labels[idx:idx+1]
    )
    axes[2, col].set_title(f'Acc: {clean_acc:.3f}', fontsize=10)
    axes[2, col].axis('off')

    axes[3, col].imshow(val_labels[idx], cmap=CLASS_CMAP,
                        vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
    axes[3, col].set_title('Labels', fontsize=10)
    axes[3, col].axis('off')

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=12, fontweight='bold', rotation=90,
                             labelpad=15)

fig.suptitle('Before / After RL Enhancement — Satellite Image Patches',
             fontsize=15, fontweight='bold', y=1.01)

legend_elements = [Patch(facecolor=CLASS_COLORS_RGB[c], label=c.replace('_', ' ').title())
                   for c in CLASSES]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.5 Land-Use Classification Maps: Before vs After ---
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
map_indices = [2, 15, 30, 50, 75]

classifier.eval()
for col, idx in enumerate(map_indices):
    # Before
    with torch.no_grad():
        x_deg = torch.from_numpy(val_degraded[idx].transpose(2,0,1)).float().unsqueeze(0).to(device)
        pred_before = classifier(x_deg).argmax(1).squeeze().cpu().numpy()
    # After
    with torch.no_grad():
        x_enh = torch.from_numpy(val_enhanced[idx].transpose(2,0,1)).float().unsqueeze(0).to(device)
        pred_after = classifier(x_enh).argmax(1).squeeze().cpu().numpy()

    gt = val_labels[idx]

    axes[0, col].imshow(pred_before, cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES-1,
                        interpolation='nearest')
    acc_b = (pred_before == gt).mean()
    axes[0, col].set_title(f'Before — Acc: {acc_b:.3f}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(pred_after, cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES-1,
                        interpolation='nearest')
    acc_a = (pred_after == gt).mean()
    axes[1, col].set_title(f'After — Acc: {acc_a:.3f}', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(gt, cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES-1,
                        interpolation='nearest')
    axes[2, col].set_title('Ground Truth', fontsize=10)
    axes[2, col].axis('off')

for row, label in enumerate(['Before Enhancement', 'After Enhancement', 'Ground Truth']):
    axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold', rotation=90, labelpad=15)

fig.suptitle('Land-Use Classification Maps — Impact of RL Enhancement',
             fontsize=15, fontweight='bold', y=1.01)
legend_elements = [Patch(facecolor=CLASS_COLORS_RGB[c], label=c.replace('_', ' ').title())
                   for c in CLASSES]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.6 Per-Class Accuracy Improvement ---
per_class_before = {c: [] for c in CLASSES}
per_class_after = {c: [] for c in CLASSES}

classifier.eval()
for idx in range(len(val_degraded)):
    gt = val_labels[idx]
    with torch.no_grad():
        x_deg = torch.from_numpy(val_degraded[idx].transpose(2,0,1)).float().unsqueeze(0).to(device)
        pred_b = classifier(x_deg).argmax(1).squeeze().cpu().numpy()
        x_enh = torch.from_numpy(val_enhanced[idx].transpose(2,0,1)).float().unsqueeze(0).to(device)
        pred_a = classifier(x_enh).argmax(1).squeeze().cpu().numpy()

    for cls_idx, cls_name in enumerate(CLASSES):
        mask = gt == cls_idx
        if mask.sum() > 0:
            per_class_before[cls_name].append((pred_b[mask] == cls_idx).mean())
            per_class_after[cls_name].append((pred_a[mask] == cls_idx).mean())

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(NUM_CLASSES)
width = 0.35

means_before = [np.mean(per_class_before[c]) for c in CLASSES]
means_after = [np.mean(per_class_after[c]) for c in CLASSES]

bars1 = ax.bar(x_pos - width/2, means_before, width, label='Before Enhancement',
               color='#EF5350', edgecolor='gray', alpha=0.85)
bars2 = ax.bar(x_pos + width/2, means_after, width, label='After Enhancement',
               color='#66BB6A', edgecolor='gray', alpha=0.85)

for bar, val in zip(bars1, means_before):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, means_after):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('_', ' ').title() for c in CLASSES], fontsize=12)
ax.set_ylabel('Classification Accuracy', fontsize=12)
ax.set_title('Per-Class Accuracy: Before vs After RL Enhancement',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\nPer-class improvement summary:')
for cls_name, bef, aft in zip(CLASSES, means_before, means_after):
    print(f'  {cls_name:12s}: {bef:.4f} → {aft:.4f}  (Δ = {aft-bef:+.4f})')

In [ ]:
# --- 6.7 Action Sequences for Different Scene Types ---
def dominant_class(label_map):
    """Return the most common class in a label map."""
    counts = np.bincount(label_map.ravel(), minlength=NUM_CLASSES)
    return CLASSES[counts.argmax()]

scene_actions = {c: [] for c in CLASSES}
for idx in range(len(val_labels)):
    dom = dominant_class(val_labels[idx])
    scene_actions[dom].append(val_action_sequences[idx])

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax_idx, cls_name in enumerate(CLASSES):
    ax = axes[ax_idx]
    action_freq = np.zeros(NUM_ACTIONS)
    for seq in scene_actions[cls_name]:
        for a in seq:
            action_freq[a] += 1
    total = action_freq.sum()
    if total > 0:
        action_freq /= total

    colors = plt.cm.Set3(np.linspace(0, 1, NUM_ACTIONS))
    ax.barh(range(NUM_ACTIONS), action_freq, color=colors, edgecolor='gray')
    ax.set_yticks(range(NUM_ACTIONS))
    ax.set_yticklabels(ACTION_NAMES, fontsize=9)
    ax.set_xlabel('Frequency', fontsize=10)
    ax.set_title(f'{cls_name.replace("_", " ").title()}-dominant',
                 fontsize=12, fontweight='bold',
                 color=CLASS_COLORS_RGB[cls_name] * 0.7)
    ax.set_xlim(0, 0.6)
    ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Action Selection Strategies by Scene Type',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.8 Accuracy Distribution: Before vs After ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(val_acc_before, bins=25, alpha=0.6, label='Before Enhancement',
        color='#EF5350', edgecolor='gray')
ax.hist(val_acc_after, bins=25, alpha=0.6, label='After Enhancement',
        color='#66BB6A', edgecolor='gray')
ax.axvline(val_acc_before.mean(), color='#B71C1C', linestyle='--', linewidth=2,
           label=f'Mean before: {val_acc_before.mean():.3f}')
ax.axvline(val_acc_after.mean(), color='#1B5E20', linestyle='--', linewidth=2,
           label=f'Mean after: {val_acc_after.mean():.3f}')

ax.set_xlabel('Pixel Classification Accuracy', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Classification Accuracy — Before vs After Enhancement',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.9 Enhancement Pipeline Walkthrough ---
example_idx = 7
state = val_env.reset(idx=example_idx)
step_images = [val_env.current_image.copy()]
step_actions = []
step_accs = [val_env._classify_accuracy(val_env.current_image)]

done = False
while not done:
    action = agent.select_action(state, training=False)
    state, _, done, info = val_env.step(action)
    step_images.append(val_env.current_image.copy())
    step_actions.append(ACTION_NAMES[action])
    step_accs.append(info['accuracy'])

n_steps = len(step_images)
fig, axes = plt.subplots(2, n_steps, figsize=(4 * n_steps, 8),
                         gridspec_kw={'height_ratios': [3, 1]})

for i in range(n_steps):
    axes[0, i].imshow(step_images[i])
    if i == 0:
        axes[0, i].set_title('Input (Degraded)', fontsize=11, fontweight='bold')
    else:
        axes[0, i].set_title(f'Step {i}: {step_actions[i-1]}', fontsize=11, fontweight='bold')
    axes[0, i].axis('off')

    color = '#4CAF50' if step_accs[i] >= step_accs[0] else '#EF5350'
    axes[1, i].bar(0, step_accs[i], color=color, width=0.5)
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_xlim(-0.5, 0.5)
    axes[1, i].set_xticks([])
    axes[1, i].text(0, step_accs[i] + 0.03, f'{step_accs[i]:.3f}',
                    ha='center', fontsize=11, fontweight='bold')
    if i == 0:
        axes[1, i].set_ylabel('Accuracy', fontsize=11)

fig.suptitle('Step-by-Step Enhancement Pipeline — RL Agent Decisions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Summary

### What We Built

A complete **RL-based satellite image enhancement pipeline** that adaptively selects processing actions to maximise downstream land-use classification accuracy.

### Key Results

| Metric | Before Enhancement | After Enhancement |
|--------|-------------------|------------------|
| Mean pixel accuracy | Baseline (degraded) | Improved by RL agent |
| Enhancement strategy | Fixed pipeline | Adaptive, scene-dependent |
| Action selection | — | Varies by dominant land-use class |

### Mathematical Foundations

- **MDP formulation**: State (image features), Actions (10 processing ops), Reward (classification-oriented)
- **NDVI proxy**: $\text{NDVI}_{\text{proxy}} = \frac{G - R}{G + R + \epsilon}$ for vegetation detection
- **Composite reward**: $R = \alpha \cdot \Delta\text{Accuracy} + \beta \cdot \text{SSIM} - \gamma \cdot \text{Distortion}$
- **DQN loss**: $\mathcal{L} = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q_{\theta^-}(s', a') - Q_\theta(s, a)\right)^2\right]$

### Key Insights

1. **Scene-adaptive processing** outperforms fixed pipelines — the RL agent learns different strategies for vegetation-heavy vs. urban-heavy vs. water-dominated scenes.
2. **Classification-driven rewards** align enhancement with the downstream task, avoiding over-processing that looks good visually but confuses the classifier.
3. **Sequential multi-step enhancement** allows the agent to compose actions (e.g., dehaze first, then sharpen) creating emergent processing pipelines.
4. **NDVI-aware features** in the state representation help the agent distinguish scene types and select appropriate spectral enhancements.

### Extensions

- **Multi-spectral support**: Extend to 4+ band imagery (NIR, SWIR) with true NDVI computation.
- **Spatial adaptation**: Apply different action sequences to different spatial regions within a single scene.
- **Actor-Critic methods**: Replace DQN with PPO/A2C for continuous action parameters (e.g., variable filter strength).
- **Real satellite data**: Fine-tune on Sentinel-2 or Landsat imagery with actual land-cover labels.
- **Multi-resolution**: Process at multiple scales and fuse results for large-area mapping.